In [1]:
import pandas as pd

def read_input(file_name, lang):
    triple_df = pd.read_json('./final/'+file_name+'.json', orient='index')
    #sort by sitelink count
    triple_df_sorted = triple_df.sort_values(by='num_sitelinks',ascending=False)

    #check if entity label in all languages
    #if True, generate prompts
    triple_df_sorted_new = triple_df_sorted[triple_df_sorted['labels_count']==8] #exclude hindi
    print(lang,':',len(triple_df_sorted_new),':',len(triple_df_sorted))
    return triple_df_sorted_new

/var/folders/9y/17y9qrrx1fz1wd3fy2ywwp740000gn/T/ipykernel_1028/225240283.py:1: DeprecationWarning: 
Pyarrow will become a required dependency of pandas in the next major release of pandas (pandas 3.0),
(to allow more performant data types, such as the Arrow string type, and better interoperability with other libraries)
but was not found to be installed on your system.
If this would cause problems for you,
please provide us feedback at https://github.com/pandas-dev/pandas/issues/54466
        
  import pandas as pd


I suggest the followin to refactor "generate_prompts" function.
The following function could be reused for P19, P569, etc., by passing different property_id and instruction_template.

In [ ]:
def generate_prompts_for_property(triple_df, ent_id_lst, lang_lst, property_id, instruction_template, sample=True):
    prompts = []
    for ent in ent_id_lst:
        ent_labels = triple_df.loc[ent, 'labels']
        if property_id in triple_df.columns and not pd.isnull(triple_df.loc[ent, property_id]):
            property_values = triple_df.loc[ent, property_id]['values']
            for value in property_values:
                for lang in lang_lst:
                    if lang in ent_labels and lang in value['labels']:
                        prompt = instruction_template.get(lang, "").format(entity=ent_labels[lang])
                        prompts.append({'entity': ent, 'language': lang, 'prompt': prompt, 'ground_truth': value['labels'][lang]})
    return prompts

In [2]:
def generate_prompts(triple_df, sample):
    lang_lst = ['ar', 'de', 'en', 'fr', 'it', 'pl', 'ru', 'zh']
    ent_id = triple_df.index
    #sampling prompts
    if sample:
        ent_id_lst = ent_id[0:100]
    else:
        ent_id_lst = ent_id
    result_df = pd.DataFrame(columns=['ent_ID','pob_ground_truth','pob_prompt','dob_ground_truth','dob_prompt','occ_ground_truth','occ_prompt','country_ground_truth','country_prompt','num_sitelinks'])

    i = 0
    for ent in ent_id_lst:
        ent_names = triple_df.loc[ent,'labels']
        result_df.loc[i,'ent_ID'] = ent
        #result dictionaries
        pob_ground_truth = {}
        pob_prompt = {}
        dob_prompt = {}
        occ_ground_truth = {}
        occ_prompt = {}
        citizen_country_ground_truth = {}
        citizen_country_prompt = {}

        for lang in lang_lst:
            #check if ent label exists in lang
            if lang in ent_names.keys():
                prompt_name = ent_names[lang]
                
                # Use : generate_prompts_for_property (triple_df, ent_id_lst, lang_lst, property_id, instruction_template, sample=True)
                # Instead of 
                
                #POB
                if not pd.isnull(triple_df.loc[ent,'P19']):
                    ent_pob = triple_df.loc[ent,'P19']['values']
                    ent_pob = ent_pob[0]['labels']           
                    if lang in ent_pob.keys():
                        match(lang):
                            case 'ar':
                                prompt = """##Instruction
                                        أجب عن السؤال بلغة السؤال.
                                        قم بالإجابة بالكلمة المفتاحية فقط. 
                                        ولا تقم بإرفاق أي تفسيرات.
                                        أعد اسم المدينة كإجابة. 
                                        ##Question 
                                        ما هو مسقط رأس """+prompt_name+""" ؟ """
                                pob_ground_truth.update({'ar' : ent_pob[lang]})
                                pob_prompt.update({'ar' : prompt})
                            case 'de':
                                prompt = """##Instruction
                                            Beantworte die Frage in der Sprache des Prompts.
                                            Gib die Antwort nur in Form eines Schlüsselworts zurück.
                                            Füge keine Erklärungen hinzu.
                                            Gib den Namen der Stadt als Antwort zurück.
                                            ##Question
                                            Was ist der Geburtsort von """+prompt_name+"""?"""
                                pob_ground_truth.update({'de' : ent_pob[lang]})
                                pob_prompt.update({'de' : prompt})
                            case 'en':
                                prompt = """##Instruction
                                            Answer the question in the prompt language. 
                                            Return only the answer keyword. 
                                            Do not include any explanations. 
                                            Return the city name as answer.  
                                            ##Question 
                                            What is the birthplace of """+prompt_name+"""?
                                            """
                                pob_ground_truth.update({'en' : ent_pob[lang]})
                                pob_prompt.update({'en' : prompt})
                            case 'fr':
                                prompt = """##Instruction
                                            Répondez à la question dans la langue de saisie. 
                                            Renvoyez uniquement la réponse brute. 
                                            N'incluez pas d'explications. 
                                            Indiquez le nom de la ville comme réponse. 
                                            ##Question 
                                            Quel est le lieu de naissance de """+prompt_name+""" ?
                                            """
                                pob_ground_truth.update({'fr' : ent_pob[lang]})
                                pob_prompt.update({'fr' : prompt})
                            case 'hi':
                                prompt = """##Instruction
                                            प्रश्न का उत्तर प्रॉम्प्ट भाषा में दें।
                                            केवल उत्तर शब्द लौटाएँ।
                                            कोई स्पष्टीकरण शामिल न करें।
                                            उत्तर के रूप में शहर का नाम लौटाएँ।
                                            ##Question
                                            """+prompt_name+""" का जन्मस्थान क्या है?
                                            """
                                pob_ground_truth.update({'hi' : ent_pob[lang]})
                                pob_prompt.update({'hi' : prompt})
                            case 'it':
                                prompt = """##Instruction
                                            Rispondi alla domanda nella stessa lingua del prompt. 
                                            Rispondi soltanto con la risposta della domanda. 
                                            Non includere commenti o spiegazioni alla risposta. 
                                            Rispondi soltanto con il paese in cui è nata la persona.
                                            ##Question 
                                            Qual è il paese di nascita di """+prompt_name+"""?
                                            """
                                pob_ground_truth.update({'it' : ent_pob[lang]})
                                pob_prompt.update({'it' : prompt})
                            case 'pl':
                                prompt = """##Instruction
                                            Odpowiedz na pytanie w języku polecenia.  
                                            Podaj wyłącznie nazwę miejscowości. 
                                            Nie dodawaj żadnych wyjaśnień. 
                                            ##Question  
                                            Gdzie urodził/urodziła się """+prompt_name+"""?
                                            """
                                pob_ground_truth.update({'pl' : ent_pob[lang]})
                                pob_prompt.update({'pl' : prompt})
                            case 'ru':
                                prompt = """##Instruction
                                            Отвечайте на вопрос на языке запроса.
                                            Возвращайте только краткий ответ в виде ключевого слова.
                                            Не добавляйте никаких объяснений.
                                            Возвращайте название города в качестве ответа.
                                            ##Question
                                            Где родился/родилась """+prompt_name+"""?
                                            """
                                pob_ground_truth.update({'ru' : ent_pob[lang]})
                                pob_prompt.update({'ru' : prompt})
                            case 'zh':
                                prompt = """##Instruction
                                            用与问题相同的语言回答给出的问题。
                                            只回答关键词答案。
                                            不要给出任何解释。
                                            返回答案必须是城市。
                                            ##Question
                                            """+prompt_name+"""的出生地是？
                                            """
                                pob_ground_truth.update({'zh' : ent_pob[lang]})
                                pob_prompt.update({'zh' : prompt})
                #DOB
                if not pd.isnull(triple_df.loc[ent,'P569']):
                    ent_dob = triple_df.loc[ent,'P569']['values']
                    ent_dob = ent_dob[0]['qid']
                    if ent_dob:
                        match(lang):
                            case 'ar':
                                prompt = """##Instruction
                                            أجب عن السؤال بلغة السؤال.
                                            قم بالإجابة بالكلمة المفتاحية فقط. 
                                            ولا تقم بإرفاق أي تفسيرات.
                                            قم بإرجاع الإجابات بصيغة ”YYYYY-MM-DD“. 
                                            ##Question
                                            ما هو تاريخ ميلاد """+prompt_name+""" ؟
                                            """
                                dob_prompt.update({'ar' : prompt})
                            case 'de':
                                prompt = """##Instruction
                                            Beantworte die Frage in der Sprache des Prompts.
                                            Gib die Antwort nur in Form eines Schlüsselworts zurück.
                                            Füge keine Erklärungen hinzu.
                                            Gib Antworten im Format ‘YYYY-MM-DD’ zurück.
                                            ##Question
                                            Was ist das Geburtsdatum von """+prompt_name+"""?
                                        """
                                dob_prompt.update({'de' : prompt})
                            case 'en':
                                prompt = """##Instruction
                                            Answer the question in the prompt language. 
                                            Return only the answer keyword. 
                                            Do not include any explanations. 
                                            Return answers in 'YYYY-MM-DD' format.  
                                            ##Question 
                                            What is the birth date of """+prompt_name+"""?
                                        """
                                dob_prompt.update({'en' : prompt})
                            case 'fr':
                                prompt = """##Instruction
                                            Répondez à la question dans la langue de saisie. 
                                            Renvoyez uniquement la réponse brute. 
                                            N'incluez pas d'explications. 
                                            Retournez les réponses au format 'YYYY-MM-DD'. 
                                            ##Question 
                                            Quelle est la date de naissance de """+prompt_name+"""?
                                        """
                                dob_prompt.update({'fr' : prompt})
                            case 'hi':
                                prompt = """##Instruction
                                            प्रश्न का उत्तर प्रॉम्प्ट भाषा में दें।
                                            केवल उत्तर शब्द लौटाएँ।
                                            कोई स्पष्टीकरण शामिल न करें।
                                            उत्तर 'YYYY-MM-DD' प्रारूप में लौटाएँ।
                                            ##Question
                                            """+prompt_name+""" की जन्म तिथि क्या है?
                                        """
                                dob_prompt.update({'hi' : prompt})
                            case 'it':
                                prompt = """##Instruction
                                            Rispondi alla domanda nella stessa lingua del prompt. 
                                            Rispondi soltanto con la risposta della domanda. 
                                            Non includere commenti o spiegazioni alla risposta. 
                                            Ritorna la risposta nel seguente formato: 'YYYY-MM-DD'. 
                                            ##Domanda
                                            Qual è la data di nascita di """+prompt_name+"""?
                                        """
                                dob_prompt.update({'it' : prompt})
                            case 'pl':
                                prompt = """
                                        ##Instruction 
                                        Odpowiedz na pytanie w języku poleceń. 
                                        Odpowiedz na pytanie podając wyłącznie datę w formacie „RRRR-MM-DD". 
                                        ##Question 
                                        Kiedy urodził/urodziła się """+prompt_name+"""?
                                        """
                                dob_prompt.update({'pl' : prompt})
                            case 'ru':
                                prompt = """##Instruction
                                            Отвечайте на вопрос на языке запроса.
                                            Возвращайте только краткий ответ в виде ключевого слова.
                                            Не добавляйте никаких объяснений.
                                            Возвращайте ответы в формате ‘YYYY-MM-DD’.
                                            ##Question
                                            Какова дата рождения """+prompt_name+"""?
                                        """
                                dob_prompt.update({'ru' : prompt})
                            case 'zh':
                                prompt = """##Instruction
                                            用与问题相同的语言回答给出的问题。
                                            只回答关键词答案。
                                            不要给出任何解释。
                                            返回答案遵循下面的形式‘YYYY-MM-DD’。
                                            ##Question
                                            """+prompt_name+"""的生日是？
                                        """
                                dob_prompt.update({'zh' : prompt})
                #Occupation
                if not pd.isnull(triple_df.loc[ent,'P106']):
                    ent_occupation = triple_df.loc[ent,'P106']['values']
                    for job in ent_occupation:
                        occupation = job['labels']
                        if lang in occupation.keys():
                            prompt_prop = occupation[lang]
                            match(lang):
                                case 'ar':
                                    prompt = """##Instruction
                                                أجب عن السؤال بلغة السؤال.
                                                قم بالإجابة بالكلمة المفتاحية فقط. 
                                                ولا تقم بإرفاق أي تفسيرات.
                                                ##Question 
                                                ما هي وظيفة """+prompt_name+""" ؟
                                                """
                                    if 'ar' in occ_ground_truth.keys():
                                        occ_ground_truth['ar'].append(prompt_prop)
                                    else:
                                        occ_ground_truth['ar'] = [prompt_prop]
                                    occ_prompt.update({'ar':prompt})
                                case 'de':
                                    prompt = """##Instruction
                                                Beantworte die Frage in der Sprache des Prompts.
                                                Gib die Antwort nur in Form eines Schlüsselworts zurück.
                                                Füge keine Erklärungen hinzu.
                                                ##Question
                                                Was ist der Beruf von """+prompt_name+"""?
                                            """
                                    if 'de' in occ_ground_truth.keys():
                                        occ_ground_truth['de'].append(prompt_prop)
                                    else:
                                        occ_ground_truth['de'] = [prompt_prop]
                                    occ_prompt.update({'de':prompt})
                                case 'en':
                                    prompt = """##Instruction
                                                Answer the question in the prompt language. 
                                                Return only the answer keyword. 
                                                Do not include any explanations.  
                                                ##Question 
                                                What is the occupation of """+prompt_name+"""?
                                            """
                                    if 'en' in occ_ground_truth.keys():
                                        occ_ground_truth['en'].append(prompt_prop)
                                    else:
                                        occ_ground_truth['en'] = [prompt_prop]
                                    occ_prompt.update({'en':prompt})
                                case 'fr':
                                    prompt = """##Instruction
                                                Répondez à la question dans la langue de saisie. 
                                                Renvoyez uniquement la réponse brute. 
                                                N'incluez pas d'explications. 
                                                ##Question 
                                                Quelle est la profession de """+prompt_name+""" ?
                                            """
                                    if 'fr' in occ_ground_truth.keys():
                                        occ_ground_truth['fr'].append(prompt_prop)
                                    else:
                                        occ_ground_truth['fr'] = [prompt_prop]
                                    occ_prompt.update({'fr':prompt})
                                case 'hi':
                                    prompt = """##Instruction
                                                प्रश्न का उत्तर प्रॉम्प्ट भाषा में दें।
                                                केवल उत्तर शब्द  लौटाएँ।
                                                कोई स्पष्टीकरण शामिल न करें।
                                                ##Question
                                                """+prompt_name+""" का व्यवसाय क्या है?
                                            """
                                    if 'hi' in occ_ground_truth.keys():
                                        occ_ground_truth['hi'].append(prompt_prop)
                                    else:
                                        occ_ground_truth['hi'] = [prompt_prop]
                                    occ_prompt.update({'hi':prompt})
                                case 'it':
                                    prompt = """##Istruzione
                                                Rispondi alla domanda nella stessa lingua del prompt. 
                                                Rispondi soltanto con la risposta della domanda. 
                                                Non includere commenti o spiegazioni alla risposta. 
                                                Rispondi solamente con l’occupazione dell’entità.
                                                ##Question 
                                                Qual è la occupazione (o il lavoro) di """+prompt_name+"""?
                                            """
                                    if 'it' in occ_ground_truth.keys():
                                        occ_ground_truth['it'].append(prompt_prop)
                                    else:
                                        occ_ground_truth['it'] = [prompt_prop]
                                    occ_prompt.update({'it':prompt})
                                case 'pl':
                                    prompt = """##Instruction
                                                Odpowiedz na pytanie w języku polecenia. 
                                                Odpowiedz jednym słowem określającym zawód. 
                                                Nie dodawaj żadnych wyjaśnień. 
                                                ##Question 
                                                Kim z zawodu jest """+prompt_name+"""?
                                            """
                                    if 'pl' in occ_ground_truth.keys():
                                        occ_ground_truth['pl'].append(prompt_prop)
                                    else:
                                        occ_ground_truth['pl'] = [prompt_prop]
                                    occ_prompt.update({'pl':prompt})
                                case 'ru':
                                    prompt = """##Instruction
                                                Отвечайте на вопрос на языке запроса.
                                                Возвращайте только краткий ответ в виде ключевого слова.
                                                Не добавляйте никаких объяснений.
                                                ##Question
                                                Какова профессия """+prompt_name+"""?
                                            """
                                    if 'ru' in occ_ground_truth.keys():
                                        occ_ground_truth['ru'].append(prompt_prop)
                                    else:
                                        occ_ground_truth['ru'] = [prompt_prop]
                                    occ_prompt.update({'ru':prompt})
                                case 'zh':
                                    prompt = """##Instruction
                                                用与问题相同的语言回答给出的问题。
                                                只回答关键词答案。
                                                不要给出任何解释。
                                                ##Question
                                                """+prompt_name+"""的职业是？
                                            """
                                    if 'zh' in occ_ground_truth.keys():
                                        occ_ground_truth['zh'].append(prompt_prop)
                                    else:
                                        occ_ground_truth['zh'] = [prompt_prop]
                                    occ_prompt.update({'zh':prompt})
                #Country of citizenship
                if not pd.isnull(triple_df.loc[ent,'P27']):
                    ent_citizen_country = triple_df.loc[ent,'P27']['values']   
                    for country in ent_citizen_country:
                        citizen_country = country['labels']
                        if lang in citizen_country.keys():
                            prompt_prop = citizen_country[lang]
                            match(lang):
                                case 'ar':
                                    prompt = """##Instruction
                                                أجب عن السؤال بلغة السؤال.
                                                قم بالإجابة بالكلمة المفتاحية فقط. 
                                                ولا تقم بإرفاق أي تفسيرات.
                                                ##Question 
                                                ما هو البلد الذي ينتمي إليه """+prompt_name+""" ؟
                                                """
                                    if 'ar' in citizen_country_ground_truth.keys():
                                        citizen_country_ground_truth['ar'].append(prompt_prop)
                                    else:
                                        citizen_country_ground_truth['ar'] = [prompt_prop]
                                    citizen_country_prompt.update({'ar':prompt})
                                case 'de':
                                    prompt = """##Instruction
                                                Beantworte die Frage in der Sprache des Prompts.
                                                Gib die Antwort nur in Form eines Schlüsselworts zurück.
                                                Füge keine Erklärungen hinzu.
                                                ##Question
                                                Was ist die Staatsangehörigkeit von """+prompt_name+"""?
                                            """
                                    if 'de' in citizen_country_ground_truth.keys():
                                        citizen_country_ground_truth['de'].append(prompt_prop)
                                    else:
                                        citizen_country_ground_truth['de'] = [prompt_prop]
                                    citizen_country_prompt.update({'de':prompt})
                                case 'en':
                                    prompt = """##Instruction
                                                Answer the question in the prompt language. 
                                                Return only the answer keyword. 
                                                Do not include any explanations. 
                                                ##Question 
                                                What is the country of citizenship of """+prompt_name+"""?
                                            """
                                    if 'en' in citizen_country_ground_truth.keys():
                                        citizen_country_ground_truth['en'].append(prompt_prop)
                                    else:
                                        citizen_country_ground_truth['en'] = [prompt_prop]
                                    citizen_country_prompt.update({'en':prompt})
                                case 'fr':
                                    prompt = """##Instruction
                                                Répondez à la question dans la langue de saisie. 
                                                Renvoyez uniquement la réponse brute. 
                                                N'incluez pas d'explications. 
                                                ##Question 
                                                Quel est le pays de citoyenneté de """+prompt_name+""" ?
                                            """
                                    if 'fr' in citizen_country_ground_truth.keys():
                                        citizen_country_ground_truth['fr'].append(prompt_prop)
                                    else:
                                        citizen_country_ground_truth['fr'] = [prompt_prop]
                                    citizen_country_prompt.update({'fr':prompt})
                                case 'hi':
                                    prompt = """##Instruction
                                                प्रश्न का उत्तर प्रॉम्प्ट भाषा में दें।
                                                केवल उत्तर शब्द लौटाएँ।
                                                कोई स्पष्टीकरण शामिल न करें।
                                                ##Question
                                                """+prompt_name+""" की नागरिकता किस देश की है?
                                            """
                                    if 'hi' in citizen_country_ground_truth.keys():
                                        citizen_country_ground_truth['hi'].append(prompt_prop)
                                    else:
                                        citizen_country_ground_truth['hi'] = [prompt_prop]
                                    citizen_country_prompt.update({'hi':prompt})
                                case 'it':
                                    prompt = """##Instruction
                                                Rispondi alla domanda nella stessa lingua del prompt. 
                                                Rispondi soltanto con la risposta della domanda. 
                                                Non includere commenti o spiegazioni alla risposta. 
                                                Rispondi solamente con il paese di cittadinanza dell’entità. 
                                                ##Question 
                                                Che cittadinanza ha """+prompt_name+"""?
                                            """
                                    if 'it' in citizen_country_ground_truth.keys():
                                        citizen_country_ground_truth['it'].append(prompt_prop)
                                    else:
                                        citizen_country_ground_truth['it'] = [prompt_prop]
                                    citizen_country_prompt.update({'it':prompt})
                                case 'pl':
                                    prompt = """##Instruction
                                                Odpowiedz na pytanie w języku polecenia. 
                                                Podaj nazwę kraju bez dodatkowych wyjaśnień. 
                                                ##Question 
                                                Jakiego państwa obywatelem jest """+prompt_name+"""?
                                            """
                                    if 'pl' in citizen_country_ground_truth.keys():
                                        citizen_country_ground_truth['pl'].append(prompt_prop)
                                    else:
                                        citizen_country_ground_truth['pl'] = [prompt_prop]
                                    citizen_country_prompt.update({'pl':prompt})
                                case 'ru':
                                    prompt = """##Instruction
                                                Отвечайте на вопрос на языке запроса.
                                                Возвращайте только краткий ответ в виде ключевого слова.
                                                Не добавляйте никаких объяснений.
                                                ##Question
                                                Какое гражданство у """+prompt_name+"""?
                                            """
                                    if 'ru' in citizen_country_ground_truth.keys():
                                        citizen_country_ground_truth['ru'].append(prompt_prop)
                                    else:
                                        citizen_country_ground_truth['ru'] = [prompt_prop]
                                    citizen_country_prompt.update({'ru':prompt})
                                case 'zh':
                                    prompt = """##Instruction
                                                用与问题相同的语言回答给出的问题。
                                                只回答关键词答案。
                                                不要给出任何解释。
                                                ##Question
                                                """+prompt_name+"""的国籍是？
                                            """
                                    if 'zh' in citizen_country_ground_truth.keys():
                                        citizen_country_ground_truth['zh'].append(prompt_prop)
                                    else:
                                        citizen_country_ground_truth['zh'] = [prompt_prop]
                                    citizen_country_prompt.update({'zh':prompt})
        
        #initialise results
        result_df.at[i,'pob_ground_truth'] = pob_ground_truth
        result_df.at[i,'pob_prompt'] = pob_prompt
        result_df.at[i,'dob_ground_truth'] = ent_dob
        result_df.at[i,'dob_prompt'] = dob_prompt
        result_df.at[i,'occ_ground_truth'] = occ_ground_truth
        result_df.at[i,'occ_prompt'] = occ_prompt
        result_df.at[i,'country_ground_truth'] = citizen_country_ground_truth
        result_df.at[i,'country_prompt'] = citizen_country_prompt
        result_df.at[i,'num_sitelinks'] = triple_df.loc[ent,'num_sitelinks']
        i = i + 1
    return result_df

In [145]:
file_names = {'ar':'Arabic_merged','zh':'Chinese_merged','en':'English_merged','fr':'French_merged','de':'German_merged','it':'Italian_merged','pl':'Polish_merged','ru':'Russian_merged'}
#only generate prompts for top 100 popular entities
sample = True
for lang, file in file_names.items():
    ground_truth_df = read_input(file, lang)
    result_df = generate_prompts(ground_truth_df, sample=True)
    #write output file
    if sample:
        result_df.to_json('./prompt/prompt_'+lang+'_100.json',indent=4, orient='records', force_ascii=False)
    else:
        result_df.to_json('./prompt/prompt_'+lang+'.json',indent=4, orient='records', force_ascii=False)


ar : 274 / 388
zh : 115 / 177
en : 3476 / 4188
fr : 374 / 491
de : 350 / 521
it : 436 / 653
pl : 137 / 231
ru : 308 / 508
